### Importing libraries

In [60]:
import pandas as pd
import numpy as np
import string
import re
import wandb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

In [61]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [62]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

### Preprocessing

In [63]:
def clean(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

In [64]:
cols = ["prompt","A","B","C","D","E"]

for col in cols:
    train[col] = train[col].apply(clean)
    test[col] = test[col].apply(clean)

In [65]:
labels = ["A","B","C","D","E"]

texts = []
targets = []
question_ids = []

In [66]:
for i in range(len(train)):
    
    prompt = train.loc[i, "prompt"]
    answer = train.loc[i, "answer"]
    
    for label in labels:
        option = train.loc[i, label]
        
        text = prompt + " [SEP] " + option
        texts.append(text)
        
        if label == answer:
            targets.append(1)
        else:
            targets.append(0)
            
        question_ids.append(i)

In [67]:
print(len(texts))

10000


### Split p=on basis of train and validation

In [68]:
question_indices = list(range(len(train)))

train_q, val_q = train_test_split(
    question_indices,
    test_size=0.2,
    random_state=42
)

In [69]:
train_texts = []
train_targets = []

val_texts = []
val_targets = []

In [70]:
for idx, qid in enumerate(question_ids):
    
    if qid in train_q:
        train_texts.append(texts[idx])
        train_targets.append(targets[idx])
    else:
        val_texts.append(texts[idx])
        val_targets.append(targets[idx])

In [71]:
print(len(train_texts))
print(len(train_targets))

print(len(val_texts))
print(len(val_targets))

8000
8000
2000
2000


### Tokenization

In [72]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

In [73]:
MAX_LEN = 128

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN
)

In [74]:
import torch
from torch.utils.data import Dataset

class MCQDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.float
        )

        return item

In [75]:
train_dataset = MCQDataset(
    train_encodings,
    train_targets
)

val_dataset = MCQDataset(
    val_encodings,
    val_targets
)

### Dataloader

In [76]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [77]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["labels"].shape)

torch.Size([64, 128])
torch.Size([64])


In [78]:
import torch.nn as nn

### BiLSTM + Attention module blocks

In [79]:
class BiLSTMAttention(nn.Module):

    def __init__(self,
                 vocab_size,
                 embed_dim=100,
                 hidden_dim=128):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.attention = nn.Linear(
            hidden_dim * 2,
            1
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(
            hidden_dim * 2,
            1
        )

    def forward(self, input_ids):

        embedded = self.embedding(input_ids)

        lstm_output, _ = self.lstm(embedded)

        attention_scores = self.attention(
            lstm_output
        ).squeeze(-1)

        attention_weights = torch.softmax(
            attention_scores,
            dim=1
        )

        context = torch.sum(
            lstm_output *
            attention_weights.unsqueeze(-1),
            dim=1
        )

        context = self.dropout(context)

        output = self.fc(context)

        return output.squeeze()

In [80]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = BiLSTMAttention(
    vocab_size=tokenizer.vocab_size
).to(device)

In [81]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [82]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb")

wandb.login(key=wandb_key)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [83]:
run = wandb.init(
    project="DL-GenAI-2026-T2",
    name="bilstm_attention_scratch",
    config={
        "model": "BiLSTM + Attention",
        "epochs": 10,
        "embedding_dim": 128,
        "hidden_dim": 128,
        "optimizer": "Adam",
        "learning_rate": 1e-3,
        "loss": "BCEWithLogitsLoss"
    }
)

epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▇▅▃▂▁▁▁▁▁
validation_accuracy,▁
validation_loss,▁
validation_map3,▁
epoch,10
train_loss,0.01368
validation_accuracy,0.994
validation_loss,0.01933
validation_map3,0.99875


### tarining

In [84]:
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()

    train_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    avg_loss = train_loss / len(train_loader)

    print(
        f"Epoch {epoch+1} "
        f"Loss: {avg_loss:.4f}"
    )

    run.log({
        "epoch": epoch + 1,
        "train_loss": avg_loss
    })

Epoch 1 Loss: 0.5134
Epoch 2 Loss: 0.4404
Epoch 3 Loss: 0.2474
Epoch 4 Loss: 0.1488
Epoch 5 Loss: 0.0977
Epoch 6 Loss: 0.0679
Epoch 7 Loss: 0.0626
Epoch 8 Loss: 0.0508
Epoch 9 Loss: 0.0452
Epoch 10 Loss: 0.0277


In [85]:
model.eval()

val_loss = 0

correct = 0
total = 0

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids)

        loss = criterion(outputs, labels)
        val_loss += loss.item()

        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

avg_val_loss = val_loss / len(val_loader)
val_accuracy = correct / total

print(f"Validation Loss: {avg_val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

run.log({
    "validation_loss": avg_val_loss,
    "validation_accuracy": val_accuracy,
})

Validation Loss: 0.0500
Validation Accuracy: 0.9820


In [86]:
model.eval()

val_probs = []

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch["input_ids"].to(device)

        outputs = model(input_ids)

        probs = torch.sigmoid(outputs)

        val_probs.extend(
            probs.cpu().numpy()
        )

In [87]:
print(len(val_probs))

2000


In [88]:
val_question_ids = []

for idx, qid in enumerate(question_ids):

    if qid not in train_q:
        val_question_ids.append(qid)

print(len(val_question_ids))

2000


In [89]:
val_answers = []

for idx, qid in enumerate(question_ids):

    if qid not in train_q:
        val_answers.append(targets[idx])

In [90]:
import numpy as np

def apk(actual, predicted, k=3):

    predicted = predicted[:k]

    score = 0.0

    for i, p in enumerate(predicted):

        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score

In [91]:
letters = ["A", "B", "C", "D", "E"]

scores = []

for i in range(0, len(val_probs), 5):

    probs = val_probs[i:i+5]

    labels = val_answers[i:i+5]

    correct_index = labels.index(1)

    actual = letters[correct_index]

    ranked = np.argsort(probs)[::-1]

    predicted = [
        letters[idx]
        for idx in ranked[:3]
    ]

    scores.append(
        apk(actual, predicted)
    )

map3 = np.mean(scores)

print(f"Validation MAP@3 : {map3:.4f}")

Validation MAP@3 : 0.9900


In [92]:
run.log({
    "validation_map3": map3
})

### Test prediction

In [93]:
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

test_texts = []
test_ids = []

for _, row in test_df.iterrows():

    for option in ["A", "B", "C", "D", "E"]:

        text = (
            row["prompt"]
            + " [SEP] "
            + row[option]
        )

        test_texts.append(text)
        test_ids.append(row["id"])

print(len(test_texts))

2500


In [94]:
test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding="max_length",
    max_length=128
)

In [95]:
class TestDataset(Dataset):

    def __init__(self, encodings):

        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):

        return {
            "input_ids": torch.tensor(
                self.encodings["input_ids"][idx]
            ),
            "attention_mask": torch.tensor(
                self.encodings["attention_mask"][idx]
            )
        }

test_dataset = TestDataset(test_encodings)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [96]:
model.eval()

test_probs = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)

        outputs = model(input_ids)

        probs = torch.sigmoid(outputs)

        test_probs.extend(
            probs.cpu().numpy()
        )

print(len(test_probs))

2500


In [97]:
letters = ["A", "B", "C", "D", "E"]

predictions = []

for i in range(0, len(test_probs), 5):

    probs = test_probs[i:i+5]

    ranked = np.argsort(probs)[::-1]

    top3 = " ".join(
        letters[idx]
        for idx in ranked[:3]
    )

    predictions.append(top3)

In [98]:
len(predictions)

500

### Submission.csv file

In [99]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

torch.save(
    model.state_dict(),
    "bilstm_attention.pt"
)

run.finish()

epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▇▄▃▂▂▂▁▁▁
validation_accuracy,▁
validation_loss,▁
validation_map3,▁
epoch,10
train_loss,0.02772
validation_accuracy,0.982
validation_loss,0.05005
validation_map3,0.99
